In [1]:
!pip install face_recognition opencv-python pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566166 sha256=cd9164c3211c094de6a26197ab8a1cfc8fdffefbae439c642076f97629739612
  Stored in directory: /root/.cache/pip/wheels/8f/47/c8/f44c5aebb7507f7c8a2c0bd23151d732d0f0bd6884ad4ac635
Successfully built face-recognition-models


In [ ]:
pip install pyngrok

In [1]:
from pyngrok import ngrok

ModuleNotFoundError: No module named 'pyngrok'

In [4]:
ngrok.set_auth_token("3BAgL8oLkqUfIR10hlzGfKgbn9W_85gPv3YVAokDsu3pxxXKV")


In [5]:
public_url = ngrok.connect(5000)
print("🔥 API URL:", public_url)

🔥 API URL: NgrokTunnel: "https://unicursal-pseudoconservatively-trinidad.ngrok-free.dev" -> "http://localhost:5000"


In [11]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import face_recognition
import numpy as np
import base64
import cv2

app = Flask(__name__)

# 🔴 SET YOUR NGROK TOKEN HERE
ngrok.set_auth_token("3BAgL8oLkqUfIR10hlzGfKgbn9W_85gPv3YVAokDsu3pxxXKV")

# Start tunnel
public_url = ngrok.connect(5000)
print("🔥 API URL:", public_url)

# Storage
known_encodings = []
known_names = []
known_usn = []

🔥 API URL: NgrokTunnel: "https://unicursal-pseudoconservatively-trinidad.ngrok-free.dev" -> "http://localhost:5000"


In [12]:
@app.route('/load', methods=['POST'])
def load_faces():
    global known_encodings, known_names, known_usn

    known_encodings.clear()
    known_names.clear()
    known_usn.clear()

    data = request.json

    if "students" not in data:
        return jsonify({"status": "error", "message": "No students provided"}), 400

    for student in data["students"]:
        try:
            img_data = base64.b64decode(student["image"].split(',')[1])
            np_arr = np.frombuffer(img_data, np.uint8)
            img = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

            enc = face_recognition.face_encodings(img)

            if len(enc) > 0:
                known_encodings.append(enc[0])
                known_names.append(student["name"])
                known_usn.append(student["usn"])

        except Exception as e:
            print("Error loading student:", str(e))

    return jsonify({
        "status": "loaded",
        "count": len(known_names)
    })

In [13]:
@app.route('/recognize', methods=['POST'])
def recognize():
    data = request.json

    if "image" not in data:
        return jsonify({"status": "error", "message": "No image provided"}), 400

    try:
        img_data = base64.b64decode(data["image"].split(',')[1])
        np_arr = np.frombuffer(img_data, np.uint8)
        img = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

        encodings = face_recognition.face_encodings(img)

        if len(encodings) == 0:
            return jsonify({"status": "no_face"})

        for enc in encodings:
            matches = face_recognition.compare_faces(known_encodings, enc)

            if True in matches:
                idx = matches.index(True)
                return jsonify({
                    "status": "success",
                    "name": known_names[idx],
                    "usn": known_usn[idx]
                })

        return jsonify({"status": "unknown"})

    except Exception as e:
        return jsonify({"status": "error", "message": str(e)})

In [14]:
@app.route('/')
def home():
    return "API is running"

In [15]:
app.run(port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 15:37:10] "POST /load HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 15:37:34] "POST /recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 15:37:36] "POST /recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 15:37:39] "POST /recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 17:06:37] "POST /load HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 17:08:02] "POST /load HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 17:20:29] "POST /recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 17:27:07] "POST /recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 17:34:26] "POST /recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Mar/2026 17:37:27] "POS